# IoT Smart Water Meter: TinyML Digit Recognition Model
### CNN Training, INT8 Quantization & C++ Header Export for ESP32-CAM
**SRS Compliance Reference:**
- **FR-02:** Edge AI Inference for mechanical dial roller digits (0–9)
- **NFR-02:** Post-training INT8 quantization to < 200 KB footprint for ESP32 SRAM
- **NFR-03:** $\ge 97.00\%$ digit classification accuracy

---
### Quick Start (Local Mac vs. Google Colab)
- **Running Locally:** Ensure you have activated the virtual environment: 
- **Running on Google Colab:** Select **Runtime > Change runtime type > T4 GPU** for rapid acceleration.


In [ ]:
# Step 1: Environment & Hardware Verification
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU Detected: {gpus}")
else:
    print("Running on CPU / Apple Silicon Metal backend.")


## Step 2: Load Synthetic Dial Dataset
The dataset consists of **100,000 augmented 28x28 grayscale images** representing both white drums (whole units) and the red drum (high-speed sub-unit #8).

In [ ]:
# Locate dataset.csv
candidates = [
    "../data/dataset.csv",
    "data/dataset.csv",
    "/content/dataset.csv"
]
dataset_path = next((p for p in candidates if os.path.exists(p)), None)

if not dataset_path:
    raise FileNotFoundError("dataset.csv not found! Please check file path.")

print(f"Loading {dataset_path}...")
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape} (100,000 samples x 785 columns)")

# Separate labels and pixel values
y = df["label"].values.astype(np.int32)
X = df.drop("label", axis=1).values.astype(np.float32) / 255.0
X = X.reshape(-1, 28, 28, 1)

print(f"X shape: {X.shape}, Pixel range: [{X.min():.1f}, {X.max():.1f}]")
print(f"Class distribution:
{pd.Series(y).value_counts().sort_index()}")


## Step 3: Exploratory Data Analysis & Sample Visualization

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    idx = np.where(y == digit)[0][0]
    ax.imshow(X[idx].squeeze(), cmap="gray")
    ax.set_title(f"Digit: {digit}", fontsize=11, fontweight="bold")
    ax.axis("off")
plt.suptitle("Sample Synthetic Water Meter Drum Digits (28x28)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Step 4: Stratified Train / Test Split (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Set:   {X_train.shape[0]:,} samples")
print(f"Validation Set: {X_test.shape[0]:,} samples")


## Step 5: Build Lightweight TinyML CNN Architecture
Designed with minimal parameters to ensure inference latency $< 100	ext{ ms}$ on the ESP32-CAM and a memory footprint well within the 200 KB limit.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu", input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation="softmax")
], name="TinyML_WaterMeter_Digit_CNN")

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## Step 6: Train the Model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)


## Step 7: Model Evaluation & Learning Curves

In [ ]:
# Evaluate Test Accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"
Test Accuracy: {test_acc * 100:.2f}% (SRS NFR-03 Target: >= 97.00%)")
assert test_acc >= 0.97, "Accuracy does not meet NFR-03 threshold!"

# Plot Training History
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history["accuracy"], label="Train Accuracy", lw=2)
ax1.plot(history.history["val_accuracy"], label="Val Accuracy", lw=2, linestyle="--")
ax1.set_title("Model Accuracy Over Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history["loss"], label="Train Loss", lw=2, color="orange")
ax2.plot(history.history["val_loss"], label="Val Loss", lw=2, color="red", linestyle="--")
ax2.set_title("Model Loss Over Epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Classification Report
y_pred = np.argmax(model.predict(X_test), axis=1)
print("
Classification Report:")
print(classification_report(y_test, y_pred))


## Step 8: Post-Training INT8 Quantization (TFLite)
Converts 32-bit floating point weights to 8-bit signed integers (), reducing memory consumption by **75%** while retaining $pprox 100\%$ accuracy.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset generator for calibration
def representative_dataset():
    for i in range(250):
        yield [X_train[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

# Save tflite file
tflite_path = "model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_quant_model)

size_kb = len(tflite_quant_model) / 1024.0
print(f"Quantized Model Size: {size_kb:.2f} KB (NFR-02 Target: < 200 KB)")
if size_kb < 200:
    print(">>> SUCCESS: Model easily fits in ESP32-CAM SRAM! <<<")


## Step 9: Export to C++ Header () for ESP32-CAM
Converts the quantized  binary into a C byte array that embeds directly into the ESP32 Arduino firmware.

In [ ]:
header_path = "model.h"
ret = os.system(f"xxd -i {tflite_path} > {header_path}")
if ret != 0 or not os.path.exists(header_path):
    with open(tflite_path, "rb") as f_in, open(header_path, "w") as f_out:
        bytes_data = f_in.read()
        f_out.write("// Auto-generated INT8 TinyML Model Header for ESP32-CAM
")
        f_out.write(f"const unsigned int model_tflite_len = {len(bytes_data)};
")
        f_out.write("const unsigned char model_tflite[] = {
  ")
        for i, b in enumerate(bytes_data):
            f_out.write(f"0x{b:02x}, ")
            if (i + 1) % 12 == 0:
                f_out.write("
  ")
        f_out.write("
};
")

print(f"C++ Header generated: {header_path} ({os.path.getsize(header_path) / 1024.0:.1f} KB)")
print("
Next Step: Copy model.h to iot/firmware/model.h and flash with Arduino IDE!")
